# 04 — Phase 1b: PF Exercise Generation — Implementation

Step-by-step verification before running the full pipeline. Generate one exercise per version, inspect output, verify structure.

## 1. Setup — load prompts + LLM client + generator

In [15]:
import sys
from pathlib import Path
import importlib
import yaml

# Project root = parent of notebooks/
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Force reload after parser fix
import src.generation.pf_generator as pfgen
importlib.reload(pfgen)
from src.generation.pf_generator import PFGenerator, PFExercise

from src.utils.llm_client import LLMClient

# Verify prompts load
prompt_path = project_root / "src" / "generation" / "prompts" / "pf_generation.yaml"
with open(prompt_path) as f:
    prompts = yaml.safe_load(f)

print("Prompt keys:", list(prompts.keys()))
print("EEDI misconceptions loaded:", len(prompts.get("EEDI_MISCONCEPTIONS", [])))
for m in prompts.get("EEDI_MISCONCEPTIONS", []):
    print(f"  - {m}")

# Use Swiss AI — Llama-3.3-70B for generation
client = LLMClient.from_role("structured")
print(f"\nModel: {client.model}")
print(f"Base URL: {client._base_url}")

# Init generator
gen = PFGenerator(client)
print("Generator initialized ✓")

Prompt keys: ['SYSTEM_PROMPT', 'USER_V1', 'USER_V2', 'USER_V3_STEP1', 'USER_V3_STEP2', 'EEDI_MISCONCEPTIONS']
EEDI misconceptions loaded: 8
  - Mixes up sides of a ratio
  - Divides total amount by each side of the ratio instead of dividing by the sum of the parts
  - Thinks the number in the ratio is the total
  - Shares equally instead of in given ratio
  - Thinks a difference of one part in a ratio means the quantities will differ by one unit
  - Does not understand that in a ratio 1:n, one part is n times bigger than the other
  - Does not understand the relationship between a ratio and a fraction
  - In ratio question where one side of ratio is given, treats that as the total

Model: meta-llama/Llama-3.3-70B-Instruct
Base URL: https://api.swissai.svc.cscs.ch/v1
Generator initialized ✓


## 2. V1 — Blind generation (grade + topic + theme only)

In [16]:
# Reload module with fixes
import importlib
import src.generation.pf_generator as pfgen
importlib.reload(pfgen)
from src.generation.pf_generator import PFGenerator, PFExercise

gen = pfgen.PFGenerator(client)

# Generate ONE V1 exercise
v1 = gen.generate_v1(
    grade="7th grade",
    topic="ratios & proportional relationships",
    temperature=0.7,
)

print(f"Version: {v1.version}")
print(f"Valid: {v1.is_valid()}")
print(f"Sections found: {list(v1.sections.keys())}")

for key, val in v1.sections.items():
    print(f"\n{'─'*50}")
    print(f"  {key}:")
    print(f"  {val[:300]}{'...' if len(val)>300 else ''}")

Version: V1
Valid: True
Sections found: ['TOPIC', 'NARRATIVE', 'DATA & CONSTRAINTS', 'TASK', 'CONSOLIDATION NOTE']

──────────────────────────────────────────────────
  TOPIC:
  ratios & proportional relationships, designing posters

──────────────────────────────────────────────────
  NARRATIVE:
  Alex is in charge of making posters for the school's campaign to reduce waste. The posters will be displayed on a large wall in the cafeteria, and Alex wants to make sure they are visually appealing and easy to read from a distance. The school has a specific font style, "Green Earth," and color sche...

──────────────────────────────────────────────────
  DATA & CONSTRAINTS:
  The sample poster is 2 feet wide with a font size of 12 points. The wall is 12 feet wide, and Alex has a limited amount of paint and ink. The school's brand guidelines specify that the font size should be proportional to the width of the poster. Alex has measured the width of the wall and calculated...

───────────────

## 3. V2 — Eedi-grounded (real misconception data)

In [17]:
# Generate ONE V2 exercise
v2 = gen.generate_v2(
    grade="7th grade",
    topic="ratios & proportional relationships",
    temperature=0.7,
)

print(f"Version: {v2.version}")
print(f"Valid: {v2.is_valid()}")
print(f"Sections found: {list(v2.sections.keys())}")
print(f"Misconceptions provided: {v2.n_misconceptions}")

for key, val in v2.sections.items():
    print(f"\n{'─'*50}")
    print(f"  {key}:")
    print(f"  {val[:300]}{'...' if len(val)>300 else ''}")

Version: V2
Valid: True
Sections found: ['TOPIC', 'NARRATIVE', 'DATA & CONSTRAINTS', 'TASK', 'CONSOLIDATION NOTE']
Misconceptions provided: 8

──────────────────────────────────────────────────
  TOPIC:
  Ratios & proportional relationships, baking cookies for a school fundraiser

──────────────────────────────────────────────────
  NARRATIVE:
  The school's baking club is making cookies for the annual fundraiser. They have decided to make two types of cookies: chocolate chip and oatmeal raisin. The club's president, Emma, wants to make sure that the cookies are distributed fairly among the club members to sell. She remembers that last year...

──────────────────────────────────────────────────
  DATA & CONSTRAINTS:
  The total number of cookies is 480, and the ratio of chocolate chip to oatmeal raisin is 3:2. The club wants to package the cookies in bags of 20. Olivia mentions that since the ratio is 3:2, they should just divide the total number of cookies by 3 to find out how many ch

## 4. V3 — LLM-inferred (two-step: infer misconceptions → generate)

In [18]:
# Generate ONE V3 exercise
v3 = gen.generate_v3(
    grade="7th grade",
    topic="ratios & proportional relationships",
    temperature=0.7,
)

print(f"Version: {v3.version}")
print(f"Valid: {v3.is_valid()}")
print(f"Sections found: {list(v3.sections.keys())}")
print(f"Misconceptions inferred: {v3.n_misconceptions}")
print(f"Inferred misconceptions:")
for m in v3.metadata.get("inferred_misconceptions", []):
    print(f"  - {m}")

for key, val in v3.sections.items():
    print(f"\n{'─'*50}")
    print(f"  {key}:")
    print(f"  {val[:300]}{'...' if len(val)>300 else ''}")

Version: V3
Valid: True
Sections found: ['TOPIC', 'NARRATIVE', 'DATA & CONSTRAINTS', 'TASK', 'CONSOLIDATION NOTE']
Misconceptions inferred: 6
Inferred misconceptions:
  - Students often believe that if two quantities are related, they must be directly proportional, ignoring the possibility of inverse or non-proportional relationships.
  - Students may think that proportional relationships only occur when the constant of proportionality is a whole number or a simple fraction, not realizing it can be any real number.
  - Some students assume that if a graph of a relationship is a straight line, it must pass through the origin, which is not necessarily true for all proportional relationships.
  - Students might confuse the concept of equivalent ratios with identical ratios, failing to recognize that equivalent ratios can have different forms but represent the same relationship.
  - Students may believe that proportions can only be used to describe linear relationships, not realizing that 

## 5. Verification — structure checks across all three

In [19]:
REQUIRED_SECTIONS = {"TOPIC", "NARRATIVE", "DATA & CONSTRAINTS", "TASK", "CONSOLIDATION NOTE"}

checks = []
for exercise in [v1, v2, v3]:
    missing = REQUIRED_SECTIONS - set(exercise.sections.keys())
    extra = set(exercise.sections.keys()) - REQUIRED_SECTIONS
    ok = len(missing) == 0
    
    # Check key PF patterns
    task_text = exercise.sections.get("TASK", "").lower()
    has_forbidden = any(w in task_text for w in ["solve for x", "calculate the", "what is the answer"])
    
    narrative = exercise.sections.get("NARRATIVE", "")
    has_narrative = len(narrative) > 50  # must be a real paragraph
    
    consolidation = exercise.sections.get("CONSOLIDATION NOTE", "")
    has_consolidation = len(consolidation) > 20
    
    checks.append({
        "version": exercise.version,
        "valid": ok,
        "missing": missing,
        "extra": extra,
        "has_narrative": has_narrative,
        "has_forbidden_task": has_forbidden,
        "has_consolidation": has_consolidation,
        "n_misconceptions": exercise.n_misconceptions,
    })

import pandas as pd
df = pd.DataFrame(checks)
print(df.to_string(index=False))
print(f"\n{'ALL VALID ✓' if all(c['valid'] for c in checks) else 'SOME INVALID — check missing sections'}")


version  valid missing extra  has_narrative  has_forbidden_task  has_consolidation  n_misconceptions
     V1   True      {}    {}           True               False               True                 0
     V2   True      {}    {}           True               False               True                 8
     V3   True      {}    {}           True               False               True                 6

ALL VALID ✓


## 6. Quick judge spot-check — verify the judge prompt works

One LLM judge call per exercise, just to verify format — NOT the full evaluation.

In [14]:
JUDGE_PROMPT = """You are an expert evaluator of Productive Failure (PF) exercises for mathematics.
Score the exercise below on 5 dimensions using a 3-point scale (1=Low, 2=Moderate, 3=High).

1. Activation of Prior Knowledge (APK):
   1 = No prerequisite concepts, 2 = One prerequisite, 3 = Two or more interacting prerequisites

2. Sweet Spot Calibration (SSC):
   1 = Too easy/hard, 2 = Moderate challenge, 3 = Prior knowledge produces plausible-but-WRONG answer

3. Affective Engagement (AE):
   1 = Abstract/textbook, 2 = Somewhat engaging, 3 = Highly engaging, relatable characters

4. Multiple Representations (MR):
   1 = Text-only, 2 = Two formats, 3 = Three or more (table, diagram, etc.)

5. Open-Endedness (OE):
   1 = Single answer/path, 2 = Some flexibility, 3 = Multiple pathways, uses "design"/"invent"

OUTPUT FORMAT:
APK: [1-3] — [1-sentence justification]
SSC: [1-3] — [1-sentence justification]
AE: [1-3] — [1-sentence justification]
MR: [1-3] — [1-sentence justification]
OE: [1-3] — [1-sentence justification]
TOTAL: [sum]/15

PF EXERCISE TO EVALUATE:
"""

# Judge ONLY the V1 exercise (spot check)
from src.utils.llm_client import LLMResponse

exercise_text = v1.raw_output
messages = [
    {"role": "system", "content": JUDGE_PROMPT},
    {"role": "user", "content": exercise_text},
]
judge_resp = client.generate(messages, max_tokens=512, temperature=0.0)
print("Judge output:")
print(judge_resp.text)
print(f"\nTokens used: {judge_resp.usage}")


Judge output:
APK: 3 — The exercise requires prior knowledge of ratios and proportional relationships, as well as an understanding of how to scale these proportions up or down, indicating the interaction of multiple prerequisite concepts.

SSC: 3 — The problem is calibrated to produce plausible-but-wrong answers, such as simply dividing the total number of each size by the package size or using the greatest common divisor to determine package sizes, which requires students to think critically about proportional scaling.

AE: 3 — The narrative is highly engaging, using relatable characters and a real-world scenario involving a basketball team and a championship game, which is likely to motivate 7th-grade students to solve the problem.

MR: 3 — The exercise involves multiple representations, including numerical data (total number of jerseys and t-shirts of each size), textual descriptions of the problem and constraints, and potentially, graphical or tabular representations that students 

## 7. Generate 5 V1 exercises — visual quality check

In [20]:
# Generate 5 V1 exercises for visual inspection
exercises = []
for i in range(5):
    ex = gen.generate_v1(
        grade="7th grade",
        topic="ratios & proportional relationships",
        temperature=0.8,  # slightly higher for variety
    )
    exercises.append(ex)
    print(f"[{i+1}/5] Valid: {ex.is_valid()} | Topic: {ex.sections.get('TOPIC', 'N/A')[:60]}")
    print(f"      Tokens: {ex.metadata.get('usage', {}).get('prompt_tokens', '?')} + {ex.metadata.get('usage', {}).get('completion_tokens', '?')}")

print("\n" + "="*60)
print("DONE — scroll down to inspect each exercise")
print("="*60)


[1/5] Valid: False | Topic: N/A
      Tokens: 1276 + 443
[2/5] Valid: True | Topic: Ratios & proportional relationships, music festival planning
      Tokens: 1276 + 540
[3/5] Valid: False | Topic: N/A
      Tokens: 1276 + 460
[4/5] Valid: True | Topic: Ratios & proportional relationships, scaling up a recipe for
      Tokens: 1276 + 1017
[5/5] Valid: True | Topic: Ratios & proportional relationships, cooking for a school fu
      Tokens: 1276 + 553

DONE — scroll down to inspect each exercise


In [22]:
# Reload with fixed parser (now handles ## Step 1: TOPIC format)
importlib.reload(pfgen)
gen_fixed = pfgen.PFGenerator(client)

# Re-parse the failed exercises
for i in [0, 2]:  # exercises 1 and 3
    ex = exercises[i]
    sections = gen_fixed._parse_sections(ex.raw_output)
    exercises[i].sections = sections
    print(f"Exercise {i+1}: {len(sections)} sections found, valid={len(sections)==5}")
    for key in sections:
        print(f"  - {key}")

# Re-verify all 5
print("\nRe-verify:")
for i, ex in enumerate(exercises):
    if ex.is_valid():
        print(f"  [{i+1}] ✓ {ex.sections.get('TOPIC', '')[0:60]}")
    else:
        missing = {"TOPIC", "NARRATIVE", "DATA & CONSTRAINTS", "TASK", "CONSOLIDATION NOTE"} - set(ex.sections.keys())
        print(f"  [{i+1}] ✗ missing: {missing}")

Exercise 1: 5 sections found, valid=True
  - TOPIC
  - NARRATIVE
  - DATA & CONSTRAINTS
  - TASK
  - CONSOLIDATION NOTE
Exercise 3: 5 sections found, valid=True
  - TOPIC
  - NARRATIVE
  - DATA & CONSTRAINTS
  - TASK
  - CONSOLIDATION NOTE

Re-verify:
  [1] ✓ Ratios & proportional relationships, baking cookies for a sc
  [2] ✓ Ratios & proportional relationships, music festival planning
  [3] ✓ Ratios & proportional relationships, recipe scaling for a sc
  [4] ✓ Ratios & proportional relationships, scaling up a recipe for
  [5] ✓ Ratios & proportional relationships, cooking for a school fu


In [ ]:
# Display exercise 2
ex = exercises[1]
print("="*70)
print(f"EXERCISE 2 | {ex.sections.get('TOPIC', 'N/A')}")
print("="*70)
for key, val in ex.sections.items():
    print(f"\n--- {key} ---")
    print(val)

In [23]:
# Display all 5 exercises
for i, ex in enumerate(exercises):
    print("="*70)
    print(f"EXERCISE {i+1} | {ex.sections.get('TOPIC', 'N/A')}")
    print("="*70)
    for key, val in ex.sections.items():
        print(f"\n--- {key} ---")
        print(val)
    print()

EXERCISE 1 | Ratios & proportional relationships, baking cookies for a school fundraiser.

--- TOPIC ---
Ratios & proportional relationships, baking cookies for a school fundraiser.

--- NARRATIVE ---
The school's baking club is making cookies for a fundraiser. They have a secret recipe that makes the perfect cookie, but they need to scale it up to make more cookies. The recipe calls for a ratio of 2 cups of flour to 1 cup of sugar. The club has 12 cups of flour and 8 cups of sugar. They want to make as many batches of cookies as possible, but they also want to make sure they use up all of the ingredients.

--- DATA & CONSTRAINTS ---
The recipe ratio is 2:1 for flour to sugar. The club has 12 cups of flour and 8 cups of sugar. If they make a batch with the exact ratio, they use 2 cups of flour and 1 cup of sugar. However, the club is considering adjusting the recipe slightly to use up all the ingredients, which might affect the taste and quality of the cookies.

--- TASK ---
Design a p

## 8. Generate 5 V2 exercises — with Eedi misconceptions

In [24]:
# Generate 5 V2 exercises (Eedi misconception-grounded)
v2_exercises = []
for i in range(5):
    ex = gen.generate_v2(
        grade="7th grade",
        topic="ratios & proportional relationships",
        temperature=0.8,
    )
    v2_exercises.append(ex)
    print(f"[{i+1}/5] Valid: {ex.is_valid()} | {ex.sections.get('TOPIC', 'N/A')[:60]}")

print("\n" + "="*60)
print("DONE")
print("="*60)


[1/5] Valid: True | Ratios & proportional relationships, making trail mix for a 
[2/5] Valid: True | Ratios & proportional relationships, school fundraiser
[3/5] Valid: True | ratios & proportional relationships, trail mix recipe
[4/5] Valid: True | Ratios & proportional relationships, baking cookies for a sc
[5/5] Valid: True | Ratios & proportional relationships, baking cookies for a sc

DONE


In [25]:
# Display all 5 V2 exercises
for i, ex in enumerate(v2_exercises):
    print("="*70)
    print(f"V2 EXERCISE {i+1} | {ex.sections.get('TOPIC', 'N/A')}")
    print(f"Misconceptions used ({ex.n_misconceptions}):")
    for m in ex.metadata.get('misconceptions', [])[:3]:
        print(f"  - {m}")
    print("="*70)
    for key, val in ex.sections.items():
        print(f"\n--- {key} ---")
        print(val)
    print()

V2 EXERCISE 1 | Ratios & proportional relationships, making trail mix for a school fundraiser
Misconceptions used (8):
  - Mixes up sides of a ratio
  - Divides total amount by each side of the ratio instead of dividing by the sum of the parts
  - Thinks the number in the ratio is the total

--- TOPIC ---
Ratios & proportional relationships, making trail mix for a school fundraiser

--- NARRATIVE ---
The school's student council is organizing a fundraiser to support the local animal shelter. They have decided to sell trail mix, a popular snack among students and teachers alike. The student council president, Emily, wants to make sure that the trail mix is not only delicious but also visually appealing. She has assigned the task of creating the perfect trail mix recipe to a team of students, including you. The team must come up with a recipe that includes nuts, dried fruits, and chocolate chips in a specific ratio.

--- DATA & CONSTRAINTS ---
The ratio of nuts to dried fruits to chocola

## 9. Test with PF philosophy added — one V1 + one V2

In [26]:
# Reload prompts with philosophy section
importlib.reload(pfgen)
gen_v2 = pfgen.PFGenerator(client)

# V1 with philosophy
v1_phil = gen_v2.generate_v1(grade="7th grade", topic="ratios & proportional relationships", temperature=0.7)
print("="*60)
print("V1 + PHILOSOPHY")
print("="*60)
for key, val in v1_phil.sections.items():
    print(f"\n--- {key} ---")
    print(val)

print("\n\n")

# V2 with philosophy
v2_phil = gen_v2.generate_v2(grade="7th grade", topic="ratios & proportional relationships", temperature=0.7)
print("="*60)
print("V2 + PHILOSOPHY")
print("="*60)
for key, val in v2_phil.sections.items():
    print(f"\n--- {key} ---")
    print(val)

V1 + PHILOSOPHY

--- TOPIC ---
Ratios & proportional relationships, music festival planning

--- NARRATIVE ---
The local music festival, "Summer Beats," is coming up, and the organizers need help planning the stage layout. The main stage will feature two popular bands, "Electric Storm" and "Rhythm Rebels." The festival organizers want to ensure that the ratio of fans for each band is reflected in the stage design. They've asked for your help in determining how to allocate the stage space. The festival organizers have conducted a survey, which shows that for every 5 fans of "Electric Storm," there are 3 fans of "Rhythm Rebels." However, the survey also reveals that "Electric Storm" fans tend to arrive in groups of 10, while "Rhythm Rebels" fans arrive in groups of 5.

--- DATA & CONSTRAINTS ---
The stage can accommodate a total of 1500 fans. The festival organizers want to allocate the space to reflect the ratio of fans for each band.

--- TASK ---
Design a stage layout that reflects th

## 10. Generate 5 V3 exercises — LLM-inferred misconceptions

In [27]:
# Generate 5 V3 exercises (LLM-inferred misconceptions, two-step)
v3_exercises = []
for i in range(5):
    ex = gen_v2.generate_v3(
        grade="7th grade",
        topic="ratios & proportional relationships",
        temperature=0.8,
    )
    v3_exercises.append(ex)
    im = ex.metadata.get('inferred_misconceptions', [])
    print(f"[{i+1}/5] Valid: {ex.is_valid()} | {ex.sections.get('TOPIC', 'N/A')[:50]} | {ex.n_misconceptions} misconceptions")

print("\n" + "="*60)
print("DONE")
print("="*60)


[1/5] Valid: True | 7th grade, ratios & proportional relationships, co | 6 misconceptions
[2/5] Valid: True | 7th grade, ratios & proportional relationships, tr | 6 misconceptions
[3/5] Valid: True | Ratios & proportional relationships, school garden | 6 misconceptions
[4/5] Valid: True | The topic for this Productive Failure (PF) exercis | 6 misconceptions
[5/5] Valid: True | Ratios & proportional relationships, planning a sc | 6 misconceptions

DONE


In [28]:
# Display all 5 V3 exercises (compact — topic + narrative + task only)
for i, ex in enumerate(v3_exercises):
    print("="*70)
    print(f"V3 EXERCISE {i+1} | {ex.sections.get('TOPIC', 'N/A')[:60]}")
    print(f"Inferred misconceptions ({ex.n_misconceptions}):")
    for m in ex.metadata.get('inferred_misconceptions', [])[:3]:
        print(f"  - {m}")
    print("="*70)
    for key, val in ex.sections.items():
        if key != "CONSOLIDATION NOTE":
            print(f"\n--- {key} ---")
            print(val)
    print()

V3 EXERCISE 1 | 7th grade, ratios & proportional relationships, cooking
Inferred misconceptions (6):
  - Students often believe that if two quantities are related, they must always increase or decrease at the same rate, without considering the concept of proportionality.
  - Many students think that ratios are only applicable when the quantities are whole numbers, and struggle to apply ratios to fractions or decimals.
  - Some students assume that equivalent ratios are the same as equal ratios, and fail to recognize that equivalent ratios can have different forms.

--- NARRATIVE ---
Tom is the manager of a school cafeteria and is in charge of preparing lunch for a festival. He needs to make a special sauce for the festival food. The recipe for the sauce involves mixing two types of ingredients, A and B, in a specific ratio. Tom has two different recipes that he thinks might work. Recipe 1 uses 3/4 cup of ingredient A and 1/2 cup of ingredient B. Recipe 2 uses 1 1/2 cups of ingredient A

## 11. V1 vs V2 — open-endedness comparison

In [29]:
# Pick best V1 (Ex 5 — music festival stage layout) and best V2 (Ex 1 — trail mix)
v1_best = exercises[4]   # V1 Ex 5: music festival stage layout
v2_best = v2_exercises[0]  # V2 Ex 1: trail mix ratio

print("="*70)
print("V1 — MUSIC FESTIVAL STAGE LAYOUT (Blind)")
print("="*70)
for key in ["TOPIC", "NARRATIVE", "DATA & CONSTRAINTS", "TASK"]:
    print(f"\n--- {key} ---")
    print(v1_best.sections.get(key, "N/A"))

print("\n\n" + "="*70)
print("V2 — TRAIL MIX RECIPE (Eedi-grounded)")
print("="*70)
for key in ["TOPIC", "NARRATIVE", "DATA & CONSTRAINTS", "TASK"]:
    print(f"\n--- {key} ---")
    print(v2_best.sections.get(key, "N/A"))

print("\n\n" + "="*70)
print("KEY DIFFERENCE:")
print("="*70)
print("""
V1 TASK: "Design a stage layout that reflects the ratio of fans for each band.
          Explain your reasoning and justify your design."

  → No correct answer. Many possible layouts. Student invents a solution.

V2 TASK: "Design a recipe for the trail mix that meets the ratio requirements
          and weighs exactly 10 pounds. Explain your reasoning and show your work."

  → One correct answer (2 lbs nuts, 3 lbs fruit, 5 lbs chocolate).
    Student computes a ratio. "Design" here means "calculate correctly."
""")

V1 — MUSIC FESTIVAL STAGE LAYOUT (Blind)

--- TOPIC ---
Ratios & proportional relationships, cooking for a school fundraiser

--- NARRATIVE ---
The 7th-grade class at Springdale Middle School is hosting a bake sale to raise funds for their upcoming field trip. They have decided to make and sell chocolate chip cookies, and they want to make sure they have enough ingredients for the big day. The recipe they're using calls for a ratio of 2 cups of flour to 1 cup of sugar. They have a large container of flour that holds 12 cups and a separate container of sugar that holds 6 cups. Emma, the class president, thinks they should just fill up both containers and use as much of each as they need, assuming they can always adjust the recipe later if needed. However, Olivia, the class treasurer, is worried that if they don't get the ratio just right, the cookies won't turn out well, and they might not sell as many. They need to figure out how much of each ingredient they should actually use to make

## Observation — SSC vs. OE tradeoff in V2

From the 15 generated exercises (5 per version), a pattern emerges:

**V1 (Blind):** Without misconceptions, the LLM sometimes produces genuinely open-ended tasks (e.g., "Design a stage layout" — no single correct answer, many possible solutions). But the data trap is weak — the numbers aren't rigged to trigger a specific wrong answer.

**V2 (Eedi-grounded):** The misconception list anchors the LLM. Misconceptions like "mixes up sides of a ratio" and "divides by each side instead of sum of parts" are all *computation errors*. The LLM naturally creates a computation problem to trigger them. The trap becomes specific ("the student will divide 10 by 2 instead of by 2+3+5=10") but the task collapses to "calculate the right ratio" — OE drops.

**Tradeoff:** Misconception grounding improves SSC (trap specificity) at the potential cost of OE (open-endedness). This mirrors the **PM-Diversity tradeoff** discovered in Phase 1a — gains in one dimension come at the expense of another. Per-dimension scoring (not a single aggregate) is essential to detect this pattern.

**Hypothesis update:** V2 > V1 on SSC and APK. V1 > V2 on OE. V3 sits between the two. The overall score may show no difference, masking a real tradeoff.